In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from warnings import filterwarnings
from sklearn.exceptions import ConvergenceWarning

from utils import get_data, train_and_plot_learning_curves

seed = 42
wine_X_tr, wine_X_val, wine_X_test, wine_y_tr, wine_y_val, wine_y_test = get_data(seed)


filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.discriminant_analysis import unique_labels


class NN(BaseEstimator, ClassifierMixin):
    def __init__(self, hidden_layer_sizes=[100], dropout=0.0, epochs=1000, batch_size=512, 
                 random_state=seed, **kwargs):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.dropout = dropout
        self.epochs = epochs
        self.batch_size = batch_size
        self.random_state = random_state
        self.kwargs = kwargs
        for key, value in kwargs.items():
            setattr(self, key, value)
        
    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        X, y = check_X_y(X, y)
        self.classes_ = unique_labels(y)
        self.num_features_ = X.shape[1]
        
        y_mapped = np.searchsorted(self.classes_, y)

        layer_dims = [self.num_features_, *self.hidden_layer_sizes, len(self.classes_)]
        layers = []
        for i in range(len(layer_dims)-2):
            layers.append(nn.Linear(layer_dims[i], layer_dims[i+1]))
            layers.append(nn.ReLU())
            if self.dropout:
                layers.append(nn.Dropout(self.dropout))
        layers.append(nn.Linear(layer_dims[-2], layer_dims[-1]))

        self.model_ = nn.Sequential(*layers)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.model_.parameters(), **self.kwargs)

        self.model_.train()
        X_tensor = torch.as_tensor(X, dtype=torch.float32)
        y_tensor = torch.as_tensor(y_mapped, dtype=torch.long)
        
        n_samples = len(X)

        for epoch in range(self.epochs):
            perm = torch.randperm(n_samples)
            X_shuffled = X_tensor[perm]
            y_shuffled = y_tensor[perm]
            
            for i in range(0, n_samples, self.batch_size):
                batch_X = X_shuffled[i : i + self.batch_size]
                batch_y = y_shuffled[i : i + self.batch_size]
                optimizer.zero_grad()
                outputs = self.model_(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()

        return self

    def predict_proba(self, X):
        check_is_fitted(self)
        X = check_array(X)

        self.model_.eval()
        with torch.inference_mode():
            X_tensor = torch.as_tensor(X, dtype=torch.float32)
            outputs = self.model_(X_tensor)
            probs = torch.softmax(outputs, dim=1)

        return probs.numpy()
    
    def predict(self, X):
        probs = self.predict_proba(X)
        return self.classes_[np.argmax(probs, axis=1)]


In [3]:
train_and_plot_learning_curves(
    [
        NN(hidden_layer_sizes=(16,)),
        NN(hidden_layer_sizes=(32,)),
        NN(hidden_layer_sizes=(64,)),
        NN(hidden_layer_sizes=(128,)),
        NN(hidden_layer_sizes=(16, 16)),
        NN(hidden_layer_sizes=(32, 32)),
        NN(hidden_layer_sizes=(64, 64)),
        NN(hidden_layer_sizes=(128, 128)),
    ],
    wine_X_tr,
    wine_y_tr,
    wine_X_val,
    wine_y_val,
    seed,
    "hidden_layer_sizes"
)

In [4]:
train_and_plot_learning_curves(
    [
        NN(hidden_layer_sizes=(128, 128), lr=0.0001),
        NN(hidden_layer_sizes=(128, 128), lr=0.0002),
        NN(hidden_layer_sizes=(128, 128), lr=0.0003),
        NN(hidden_layer_sizes=(128, 128), lr=0.0004),
        NN(hidden_layer_sizes=(128, 128), lr=0.0005),
        NN(hidden_layer_sizes=(128, 128), lr=0.0006),
        NN(hidden_layer_sizes=(128, 128), lr=0.0007),
        NN(hidden_layer_sizes=(128, 128), lr=0.0008),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009),
        NN(hidden_layer_sizes=(128, 128), lr=0.001),
    ],
    wine_X_tr,
    wine_y_tr,
    wine_X_val,
    wine_y_val,
    seed,
    "lr"
)

In [5]:
train_and_plot_learning_curves(
    [
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, weight_decay=0),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, weight_decay=0.00001),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, weight_decay=0.00002),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, weight_decay=0.00003),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, weight_decay=0.00004),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, weight_decay=0.00005),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, weight_decay=0.00006),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, weight_decay=0.00007),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, weight_decay=0.00008),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, weight_decay=0.00009),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, weight_decay=0.0001),
    ],
    wine_X_tr,
    wine_y_tr,
    wine_X_val,
    wine_y_val,
    seed,
    "weight_decay"
)

In [ ]:
train_and_plot_learning_curves(
    [
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, dropout=0),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, dropout=0.01),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, dropout=0.02),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, dropout=0.03),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, dropout=0.04),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, dropout=0.05),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, dropout=0.06),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, dropout=0.07),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, dropout=0.08),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, dropout=0.09),
        NN(hidden_layer_sizes=(128, 128), lr=0.0009, dropout=0.1),
    ],
    wine_X_tr,
    wine_y_tr,
    wine_X_val,
    wine_y_val,
    seed,
    "dropout"
)

In [24]:
best_model = NN(dropout=0.3, hidden_layer_sizes=(128,))
best_model.fit(wine_X_tr, wine_y_tr)
best_model.score(wine_X_test, wine_y_test)

0.5646153846153846

In [ ]:
autogluon_nn = NN(
    hidden_layer_sizes=[128] * 4,
    dropout=0.1,
    epochs=1000,
    batch_size=512,
    early_stopping=False,
    lr=0.0003,
    weight_decay=1e-6,
)
autogluon_nn.fit(wine_X_tr, wine_y_tr)
print(f"Train accuracy: {autogluon_nn.score(wine_X_tr, wine_y_tr):.4f}")
print(f"Test accuracy:   {autogluon_nn.score(wine_X_test, wine_y_test):.4f}")

Train accuracy: 0.6184
Val accuracy:   0.5692
